# UI Element Detector - Colab training

Two stages, same split and seed:
1. **YOLO26n** detects element boxes on the full screenshot.
2. **DenseNet121, Xception41, SE-ResNet50, PyramidNet-110, Residual Attention Network** classify the detected crop.

Runtime: `Runtime > Change runtime type > T4 GPU`.

Before running: put `ui-data.zip` (the `data/yolo` folder + `results` skeleton) in your Google Drive root, or upload it, and set `ZIP_PATH` below.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/ui-data.zip'   # <- change if needed
REPO_URL = 'https://github.com/AnaraHayat/ui-Element-detector.git'

!git clone -q {REPO_URL} /content/project || true
%cd /content/project
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, pathlib
os.makedirs('data', exist_ok=True)
if not pathlib.Path('data/yolo/data.yaml').exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall('data')
print(open('data/yolo/data.yaml').read())
!ls data/yolo/images/train | head -3
!ls data/yolo/images/train | wc -l

In [ ]:
# make data.yaml portable: drop the absolute Windows path written on the local machine,
# otherwise Ultralytics looks for D:/python/... and cannot find the images
import pathlib, yaml
p = pathlib.Path('data/yolo/data.yaml')
cfg = yaml.safe_load(p.read_text())
if cfg.pop('path', None) is not None:
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('removed absolute path from data.yaml')
print(p.read_text())

In [ ]:
# dataset sanity: paths resolve, 15 classes, images found
from ultralytics.data.utils import check_det_dataset
d = check_det_dataset('data/yolo/data.yaml')
print('train:', d['train'], '| nc:', d['nc'])
print(list(d['names'].values()))

In [ ]:
# 2-minute smoke test before spending GPU hours
!python scripts/models/check_models.py --size 128
!python scripts/training/train_cnn.py --model densenet121 --epochs 1 --max-crops 500 --val-per-class 20 --batch-size 32 --out-suffix _smoke

In [ ]:
# stage 1: five classifiers (each run is resumable, results go to results/<model>/)
for m in ['densenet121', 'xception41', 'seresnet50', 'pyramidnet110', 'residual_attention']:
    print('=' * 70); print('TRAINING', m)
    !python scripts/training/train_cnn.py --model {m}
    !cp results/{m}/best.pt /content/drive/MyDrive/models_{m}_best.pt

In [ ]:
# stage 2: detector
!python scripts/training/train_yolo.py --model yolo26n
!cp results/yolo26n/weights/best.pt /content/drive/MyDrive/models_yolo26n_best.pt

In [ ]:
# package all logs/metrics for download
import shutil
shutil.make_archive('/content/drive/MyDrive/ui-results', 'zip', 'results')
print('done -> ui-results.zip')